In [25]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# This tells Plotly to skip the notebook and just open a browser tab
import plotly.io as pio
pio.renderers.default = "browser"

analyzer = SentimentIntensityAnalyzer()

In [26]:
# Load local data
news_df = pd.read_csv('../data/raw_headlines.csv')
price_df = pd.read_csv('../data/stock_prices.csv')

# Preprocessing
news_df['Date'] = pd.to_datetime(news_df['Date'])
price_df['Date'] = pd.to_datetime(price_df['Date'])

# Calculate Sentiment and group by day
news_df['Sentiment_Score'] = news_df['Headline'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
daily_mood = news_df.groupby('Date')['Sentiment_Score'].mean().reset_index()

# Merge into a final analysis dataframe
df = pd.merge(price_df, daily_mood, on='Date', how='inner')
df['Price_Return'] = df['Close'].pct_change()

In [27]:
# Keywords to check
keywords = ['earnings', 'growth', 'crash', 'warns', 'rally', 'downgrade']

# Create binary columns for keywords
for word in keywords:
    df[word] = df['Date'].apply(lambda d: 1 if news_df[
        (news_df['Date'] == d) &
        (news_df['Headline'].str.lower().str.contains(word))
    ].shape[0] > 0 else 0) # Added [0] index for correct shape check

# Calculate Correlation
corr_columns = [w for w in keywords if w in df.columns] + ['Price_Return']
corr_data = df[corr_columns].corr().fillna(0) # Fill NaNs with 0 to prevent empty plots

# Plot Heatmap
fig_heat = px.imshow(
    corr_data,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title="Correlation: News Keywords vs. Price Movement"
)
fig_heat.update_layout(font_family="Times New Roman", template="plotly_dark")
fig_heat.show()

In [28]:
fig_trend = make_subplots(specs=[[{"secondary_y": True}]])

fig_trend.add_trace(go.Scatter(x=df['Date'], y=df['Close'], name="Price"), secondary_y=False)
fig_trend.add_trace(go.Bar(x=df['Date'], y=df['Sentiment_Score'], name="Sentiment", opacity=0.3), secondary_y=True)

fig_trend.update_layout(title="Daily Mood vs. Stock Price",font_family="Times New Roman", template="plotly_dark")
fig_trend.show()

In [29]:
fig_candle = go.Figure(data=[go.Candlestick(
    x=df['Date'], open=df['Open'], high=df['High'], low=df['Low'], close=df['Close']
)])

# Example Marker: Buy if Sentiment > 0.4
buys = df[df['Sentiment_Score'] > 0.4]
fig_candle.add_trace(go.Scatter(x=buys['Date'], y=buys['Low']*0.98, mode='markers',
                                marker=dict(symbol='triangle-up', size=12, color='lime'), name='AI Buy'))

fig_candle.update_layout(title="Trading Chart with Sentiment Signals", font_family="Times New Roman", template="plotly_dark")
fig_candle.show()